In [ ]:
import random, math, csv
import numpy as np
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import plotly.graph_objects as go
from torch.nn.utils import spectral_norm

SEED        = 42
EPOCHS      = 80
BATCH_SIZE  = 128

LR           = 1e-3
WEIGHT_DECAY = 1e-4

ALPHA_CE     = 1.0
HINGE_WEIGHT = 1.0
RECON_WEIGHT = 0.1

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PERSISTENT  = False
PIN_MEMORY  = (DEVICE == "cuda")

MARGIN        = 1.0
SIGMA_Q       = 0.4
ALPHA         = 0.2
CLS_SCALE     = 1.0

SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 1.5
N_SIGMA_GRID   = 11
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

NUM_CLASSES = 10
LATENT_DIM  = 128

SMOOTH_WINDOW_EPOCHS = 3

def set_seed(seed: int = 43):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

class AELinearCIFAR10(nn.Module):
    def __init__(self, latent_dim: int = LATENT_DIM, num_classes: int = NUM_CLASSES):
        super().__init__()

        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.flatten = nn.Flatten()
        self.enc_fc = nn.Sequential(
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, latent_dim),
        )

        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 128 * 8 * 8),
            nn.ReLU(inplace=True),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 3, kernel_size=3, stride=1, padding=1),
        )

        self.classifier = spectral_norm(nn.Linear(latent_dim, num_classes))

    def encode(self, x):
        h = self.encoder_conv(x)
        h = self.flatten(h)
        z = self.enc_fc(h)
        return z

    def decode(self, z):
        h = self.dec_fc(z)
        h = h.view(-1, 128, 8, 8)
        x_rec = self.decoder_conv(h)
        return x_rec

    def forward(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        x_rec = self.decode(z)
        return CLS_SCALE * logits, x_rec

    @torch.no_grad()
    def logits_from_x_nograd(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        return CLS_SCALE * logits

    @torch.no_grad()
    def features_from_x_nograd(self, x):
        return self.encode(x)

def cross_entropy_from_logits(logits, y, label_smoothing: float = 0.0):
    return F.cross_entropy(logits, y, reduction='mean', label_smoothing=label_smoothing)

def margin_hinge_loss_from_logits(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    v = F.relu(margin - m)
    return v.mean()

@torch.no_grad()
def samplewise_margin_and_hinge(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    h = F.relu(margin - m)
    return m, h

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_test  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
test_set  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

def make_half_splits(dataset, batch_size=128):
    n = len(dataset)
    idx_cal = np.arange(0, n//2)
    idx_tst = np.arange(n//2, n)
    ds_cal  = Subset(dataset, idx_cal)
    ds_tst  = Subset(dataset, idx_tst)
    ld_cal  = DataLoader(ds_cal, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    ld_tst  = DataLoader(ds_tst, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    return ld_cal, ld_tst

P_cal_loader, P_tst_loader = make_half_splits(test_set, BATCH_SIZE)
Q_cal_loader, Q_tst_loader = P_cal_loader, P_tst_loader

def apply_Q_transform(x: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0.0:
        return x

    B = x.size(0)
    device = x.device

    contrast_scale = 1.0 + 0.8 * sigma * (2.0 * torch.rand(B, 3, 1, 1, device=device) - 1.0)
    brightness_shift = 0.4 * sigma * (2.0 * torch.rand(B, 3, 1, 1, device=device) - 1.0)
    x = x * contrast_scale + brightness_shift

    max_shift = int(4 * sigma)
    if max_shift > 0:
        shift_x = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        shift_y = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        for i in range(B):
            x[i] = torch.roll(x[i], shifts=(int(shift_y[i]), int(shift_x[i])), dims=(1, 2))

    noise_std = 0.1 * sigma
    if noise_std > 0:
        x = x + noise_std * torch.randn_like(x)

    return x

model = AELinearCIFAR10(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = None

def train_one_epoch(epoch: int):
    model.train()
    sums = {'cce':0.0,'hinge':0.0,'recon':0.0,'acc':0.0,'n':0}
    pbar = tqdm(DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT),
                desc=f"Train {epoch:02d}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, x_rec = model(x)
        cce   = cross_entropy_from_logits(logits, y, label_smoothing=0.0)
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN)
        recon = F.mse_loss(x_rec, x)

        loss = ALPHA_CE * cce + HINGE_WEIGHT * hinge + RECON_WEIGHT * recon

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        sums['cce']   += cce.item()*bs
        sums['hinge'] += hinge.item()*bs
        sums['recon'] += recon.item()*bs
        sums['acc']   += acc*bs
        sums['n']     += bs
        pbar.set_postfix({
            'cce':   f"{sums['cce']/sums['n']:.4f}",
            'hinge': f"{sums['hinge']/sums['n']:.4f}",
            'recon': f"{sums['recon']/sums['n']:.4f}",
            'acc':   f"{sums['acc']/sums['n']:.4f}",
        })
    for k in ['cce','hinge','recon','acc']:
        sums[k] /= sums['n']
    return sums

@torch.no_grad()
def evaluate(loader, tag="Eval"):
    model.eval()
    tot = {'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(loader, desc=f"{tag}", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        cce   = F.cross_entropy(logits, y, reduction='mean').item()
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN).item()
        acc   = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        tot['cce']  += cce*bs
        tot['hinge']+= hinge*bs
        tot['acc']  += acc*bs
        tot['n']    += bs
    for k in ['cce','hinge','acc']:
        tot[k] /= tot['n']
    return tot

@torch.no_grad()
def predictive_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(logits, dim=1)
    log_probs = (probs + 1e-12).log()
    ent = -(probs * log_probs).sum(dim=1)
    return ent

@torch.no_grad()
def scores_for_labels_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> np.ndarray:
    B = logits.size(0)
    idx = torch.arange(B, device=logits.device)
    masked = logits.clone()
    masked[idx, labels] = -float('inf')
    max_other = masked.max(dim=1).values
    s = (max_other - logits[idx, labels]).detach().float().cpu().numpy()
    return s

def conformal_quantile(scores: np.ndarray, alpha: float, finite_sample: bool = True) -> float:
    n = scores.size
    if n == 0:
        return float('nan')
    if finite_sample:
        qlevel = math.ceil((n + 1) * (1 - alpha)) / n
    else:
        qlevel = 1 - alpha
    if qlevel > 1:
        return float('inf')
    try:
        return float(np.quantile(scores, qlevel, method='higher'))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation='higher'))

def coverage_given_threshold(scores_true: np.ndarray, threshold: float) -> float:
    if scores_true.size == 0:
        return float('nan')
    return float((scores_true <= threshold).mean())

@torch.no_grad()
def collect_true_scores(loader: DataLoader, sigma: float) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        s = scores_for_labels_from_logits(logits, y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def expected_set_size_threshold(loader: DataLoader, sigma: float, q: float) -> float:
    model.eval()
    total_size = 0.0
    total_n = 0
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        B, K = logits.shape

        values, indices = logits.topk(k=2, dim=1)
        max1 = values[:, 0:1]
        max2 = values[:, 1:2]
        argmax = indices[:, 0]

        classes = torch.arange(K, device=logits.device).unsqueeze(0).expand(B, -1)
        argmax_exp = argmax.unsqueeze(1).expand(-1, K)
        max_other = torch.where(classes == argmax_exp, max2, max1)

        scores_all = max_other - logits
        set_sizes = (scores_all <= q).sum(dim=1).float()

        total_size += set_sizes.sum().item()
        total_n += B

    return total_size / total_n if total_n > 0 else float('nan')

@torch.no_grad()
def collect_pseudo_scores_mixed(
    loader: DataLoader,
    sigma: float,
    u: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        yhat = logits.argmax(dim=1)

        B, K = logits.shape
        rand_labels = torch.randint(low=0, high=num_classes, size=(B,), device=logits.device)
        mask_low = (ent <= u)
        pseudo_y = torch.where(mask_low, yhat, rand_labels)

        s = scores_for_labels_from_logits(logits, pseudo_y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def compute_uP_star(
    P_cal_loader: DataLoader,
    P_tst_loader: DataLoader,
    alpha: float = ALPHA,
    num_u_points: int = 30,
):
    ent_all = []
    for x, _ in P_cal_loader:
        x = x.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        ent_all.append(ent.cpu().numpy())
    ent_all = np.concatenate(ent_all, axis=0)

    low_q, high_q = 0.05, 0.95
    t_min = np.quantile(ent_all, low_q)
    t_max = np.quantile(ent_all, high_q)
    u_cutoffs = np.linspace(t_min, t_max, num_u_points)

    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
    cov_P_vals = []
    target_cov = 1.0 - alpha

    u_P_star = u_cutoffs[0]
    q_Phat_uP_star = None

    for u in u_cutoffs:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u,
            num_classes=NUM_CLASSES,
        )
        q_Phat_u = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        cov_P_u = coverage_given_threshold(scores_P_tst_true, q_Phat_u)
        cov_P_vals.append(cov_P_u)

        if cov_P_u >= target_cov:
            u_P_star = u
            q_Phat_uP_star = q_Phat_u

    cov_P_vals = np.array(cov_P_vals)

    if q_Phat_uP_star is None:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u_cutoffs[0],
            num_classes=NUM_CLASSES,
        )
        q_Phat_uP_star = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        u_P_star = float(u_cutoffs[0])

    return float(u_P_star), u_cutoffs, cov_P_vals, float(q_Phat_uP_star)

hinge_P_epochs         = np.zeros(EPOCHS, dtype=np.float64)
cov_Q_uP_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
cov_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
u_P_star_epochs        = np.zeros(EPOCHS, dtype=np.float64)

ESS_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
ESS_P_oracle_epochs    = np.zeros(EPOCHS, dtype=np.float64)
ESS_Q_uP_epochs_sigma      = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
ESS_Q_oracle_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)

print("=== Training with per-epoch source-tuned pseudo-label coverage & set-size tracking (Conv AE + Linear, CIFAR-10) ===")

for epoch in range(1, EPOCHS+1):
    tr = train_one_epoch(epoch)
    teP = evaluate(P_tst_loader, tag=f"Test(P) epoch {epoch}")
    hinge_P = teP["hinge"]
    hinge_P_epochs[epoch-1] = hinge_P

    u_P_star, u_grid, cov_P_over_u, q_Phat_uP_star = compute_uP_star(
        P_cal_loader=P_cal_loader,
        P_tst_loader=P_tst_loader,
        alpha=ALPHA,
        num_u_points=30,
    )
    u_P_star_epochs[epoch-1] = u_P_star

    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
    cov_P_uP_star = coverage_given_threshold(scores_P_tst_true, q_Phat_uP_star)
    cov_P_uP_epochs[epoch-1] = cov_P_uP_star

    scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
    q_P_true_oracle = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

    ESS_P_uP = expected_set_size_threshold(P_tst_loader, sigma=0.0, q=q_Phat_uP_star)
    ESS_P_oracle = expected_set_size_threshold(P_tst_loader, sigma=0.0, q=q_P_true_oracle)
    ESS_P_uP_epochs[epoch-1] = ESS_P_uP
    ESS_P_oracle_epochs[epoch-1] = ESS_P_oracle

    print(
        f"Epoch {epoch:03d} | "
        f"Train CE {tr['cce']:.4f} hinge {tr['hinge']:.4f} recon {tr['recon']:.4f} acc {tr['acc']:.4f} | "
        f"Test(P) CE {teP['cce']:.4f} hinge {teP['hinge']:.4f} acc {teP['acc']:.4f} | "
        f"u_P^*={u_P_star:.4f}, Cov_P(u_P^*)={cov_P_uP_star:.4f}, "
        f"ESS_P(u_P^*)={ESS_P_uP:.2f}, ESS_P(oracle)={ESS_P_oracle:.2f}"
    )

    if scheduler is not None:
        scheduler.step()

    for s_idx, sigma in enumerate(SIGMA_GRID):
        sigma_val = float(sigma)

        scores_Q_cal_pseudo = collect_pseudo_scores_mixed(
            loader=Q_cal_loader,
            sigma=sigma_val,
            u=u_P_star,
            num_classes=NUM_CLASSES,
        )
        q_Qhat_uP_sigma = conformal_quantile(scores_Q_cal_pseudo, ALPHA, finite_sample=True)

        scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma=sigma_val)
        q_Q_true_sigma = conformal_quantile(scores_Q_cal_true, ALPHA, finite_sample=True)

        scores_Q_tst_true = collect_true_scores(
            loader=Q_tst_loader,
            sigma=sigma_val,
        )
        cov_Q_uP = coverage_given_threshold(scores_Q_tst_true, q_Qhat_uP_sigma)
        cov_Q_uP_epochs_sigma[epoch-1, s_idx] = cov_Q_uP

        ESS_Q_uP = expected_set_size_threshold(Q_tst_loader, sigma=sigma_val, q=q_Qhat_uP_sigma)
        ESS_Q_oracle = expected_set_size_threshold(Q_tst_loader, sigma=sigma_val, q=q_Q_true_sigma)
        ESS_Q_uP_epochs_sigma[epoch-1, s_idx] = ESS_Q_uP
        ESS_Q_oracle_epochs_sigma[epoch-1, s_idx] = ESS_Q_oracle

    if SIGMA_MIN_GRID <= SIGMA_Q <= SIGMA_MAX_GRID:
        s_idx_print = int(
            round(
                (SIGMA_Q - SIGMA_MIN_GRID)
                / (SIGMA_MAX_GRID - SIGMA_MIN_GRID)
                * (N_SIGMA_GRID - 1)
            )
        )
        cov_Q_at_SIGMAQ = cov_Q_uP_epochs_sigma[epoch-1, s_idx_print]
        ESS_Q_uP_at_SIGMAQ = ESS_Q_uP_epochs_sigma[epoch-1, s_idx_print]
        ESS_Q_oracle_at_SIGMAQ = ESS_Q_oracle_epochs_sigma[epoch-1, s_idx_print]
        print(
            f"  Q at σ={SIGMA_Q:.2f} using u_P^*: "
            f"Coverage={cov_Q_at_SIGMAQ:.4f}, "
            f"ESS_Q(u_P^*)={ESS_Q_uP_at_SIGMAQ:.2f}, "
            f"ESS_Q(oracle)={ESS_Q_oracle_at_SIGMAQ:.2f}"
        )

def smooth_1d(a: np.ndarray, window: int = 3) -> np.ndarray:
    a = np.asarray(a, dtype=np.float64)
    if window <= 1 or a.size == 0 or a.size < window:
        return a.copy()
    pad_left = window // 2
    pad_right = window - 1 - pad_left
    a_pad = np.pad(a, (pad_left, pad_right), mode="edge")
    kernel = np.ones(window, dtype=np.float64) / float(window)
    return np.convolve(a_pad, kernel, mode="valid")

def smooth_2d_along_epochs(Z: np.ndarray, window: int = 3) -> np.ndarray:
    Z = np.asarray(Z, dtype=np.float64)
    if window <= 1 or Z.size == 0 or Z.shape[0] < window:
        return Z.copy()
    Z_sm = np.empty_like(Z)
    for j in range(Z.shape[1]):
        Z_sm[:, j] = smooth_1d(Z[:, j], window)
    return Z_sm

hinge_P_plot     = smooth_1d(hinge_P_epochs, window=SMOOTH_WINDOW_EPOCHS)
u_P_star_plot    = smooth_1d(u_P_star_epochs, window=SMOOTH_WINDOW_EPOCHS)
cov_Q_uP_plot    = smooth_2d_along_epochs(cov_Q_uP_epochs_sigma, window=SMOOTH_WINDOW_EPOCHS)
ESS_Q_uP_plot    = smooth_2d_along_epochs(ESS_Q_uP_epochs_sigma, window=SMOOTH_WINDOW_EPOCHS)

print("Smoothing done for plotting (window =", SMOOTH_WINDOW_EPOCHS, ")")

LOSS_AXIS  = hinge_P_plot
SIGMA_AXIS = SIGMA_GRID
SIGMA_MESH, LOSS_MESH = np.meshgrid(SIGMA_AXIS, LOSS_AXIS, indexing="ij")
Z_cov = cov_Q_uP_plot.T

nominal_cov = 1.0 - ALPHA
Z_nom = nominal_cov * np.ones_like(Z_cov)

fig3d_cov = go.Figure()
fig3d_cov.add_surface(
    x=SIGMA_MESH, y=LOSS_MESH, z=Z_cov,
    colorscale="Viridis", opacity=0.9,
    name="Coverage on Q using u_P^*", showscale=True,
)
fig3d_cov.add_surface(
    x=SIGMA_MESH, y=LOSS_MESH, z=Z_nom,
    colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
    opacity=0.3, name="Nominal 1−α plane", showscale=False,
)
fig3d_cov.update_layout(
    title="CIFAR-10 (Conv AE + linear): Coverage on Q vs (σ, hinge loss on P)\nSource-tuned mixed pseudo-labeling (u_P^*)",
    scene=dict(
        xaxis_title="σ (Q shift level)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Coverage on Q",
        zaxis=dict(range=[0.0, 1.02]),
    ),
    width=900, height=700,
)
fig3d_cov.write_html("cifar10_ae_linear_uP_coverage_vs_sigma_hinge_3d.html", include_plotlyjs="cdn")
print("Saved 3D coverage plot to cifar10_ae_linear_uP_coverage_vs_sigma_hinge_3d.html")

Z_ess_Q_uP = ESS_Q_uP_plot.T
fig3d_ess = go.Figure()
fig3d_ess.add_surface(
    x=SIGMA_MESH, y=LOSS_MESH, z=Z_ess_Q_uP,
    colorscale="Viridis", opacity=0.9,
    name="Expected set size on Q using u_P^*", showscale=True,
)
fig3d_ess.update_layout(
    title="CIFAR-10 (Conv AE + linear): Expected set size on Q vs (σ, hinge loss on P)\nSource-tuned mixed pseudo-labeling (u_P^*)",
    scene=dict(
        xaxis_title="σ (Q shift level)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Expected set size on Q",
        zaxis=dict(range=[0.5, NUM_CLASSES + 0.5]),
    ),
    width=900, height=700,
)
fig3d_ess.write_html("cifar10_ae_linear_uP_exp_set_size_vs_sigma_hinge_3d.html", include_plotlyjs="cdn")
print("Saved 3D expected set size plot to cifar10_ae_linear_uP_exp_set_size_vs_sigma_hinge_3d.html")

fig_u_vs_hinge = go.Figure()
fig_u_vs_hinge.add_trace(
    go.Scatter(
        x=hinge_P_plot,
        y=u_P_star_plot,
        mode="markers+lines",
        name="u_P^* vs hinge(P)"
    )
)
fig_u_vs_hinge.update_layout(
    title="CIFAR-10 (Conv AE + linear): u_P^* vs hinge loss on P_tst (smoothed)",
    xaxis_title="Hinge loss on P_tst",
    yaxis_title="u_P^* (entropy cutoff)",
    width=800, height=500,
)
fig_u_vs_hinge.write_html("cifar10_ae_linear_uP_vs_hingeP.html", include_plotlyjs="cdn")
print("Saved u_P^* vs hinge(P) plot to cifar10_ae_linear_uP_vs_hingeP.html")

csv_path = "cifar10_ae_linear_uP_expected_set_sizes.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "domain", "sigma", "hinge_P",
                     "ESS_uP", "ESS_oracle"])
    for epoch in range(1, EPOCHS+1):
        hinge = hinge_P_epochs[epoch-1]
        writer.writerow([
            epoch, "P", 0.0, hinge,
            ESS_P_uP_epochs[epoch-1],
            ESS_P_oracle_epochs[epoch-1],
        ])
        for s_idx, sigma in enumerate(SIGMA_GRID):
            writer.writerow([
                epoch, "Q", float(sigma), hinge,
                ESS_Q_uP_epochs_sigma[epoch-1, s_idx],
                ESS_Q_oracle_epochs_sigma[epoch-1, s_idx],
            ])

print(f"Saved expected set size table to {csv_path}")

print("\nDone. Arrays available (raw, unsmoothed):")
print("hinge_P_epochs.shape =", hinge_P_epochs.shape)
print("cov_Q_uP_epochs_sigma.shape =", cov_Q_uP_epochs_sigma.shape)
print("cov_P_uP_epochs.shape =", cov_P_uP_epochs.shape)
print("u_P_star_epochs.shape =", u_P_star_epochs.shape)
print("ESS_P_uP_epochs.shape =", ESS_P_uP_epochs.shape)
print("ESS_P_oracle_epochs.shape =", ESS_P_oracle_epochs.shape)
print("ESS_Q_uP_epochs_sigma.shape =", ESS_Q_uP_epochs_sigma.shape)
print("ESS_Q_oracle_epochs_sigma.shape =", ESS_Q_oracle_epochs_sigma.shape)

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scienceplots
import matplotlib as mpl

plt.style.use(["science", "ieee", "no-latex"])

mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

IEEE_WIDTH_1COL = 5.0
IEEE_ASPECT = 0.9
FIGSIZE_1COL = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

@torch.no_grad()
def collect_pseudo_scores_hard(
    loader: DataLoader,
    sigma: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)
        s = scores_for_labels_from_logits(logits, yhat)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def extract_features_batch(x: torch.Tensor) -> torch.Tensor:
    model.eval()
    return model.features_from_x_nograd(x)

@torch.no_grad()
def estimate_feature_rho_mix(
    loader: DataLoader,
    sigma: float,
    max_batches: int = None,
    trim_quantile: float = 0.9,
):
    model.eval()
    all_dists = []

    for b, (x, _) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break

        x = x.to(DEVICE)
        z = extract_features_batch(x)
        x_shift = apply_Q_transform(x.clone(), sigma)
        z_shift = extract_features_batch(x_shift)

        diff = z - z_shift
        d = diff.norm(p=2, dim=1)
        all_dists.append(d.detach().cpu().numpy())

    if not all_dists:
        return float("nan"), float("nan")

    dists = np.concatenate(all_dists, axis=0)
    rho_full = float(dists.mean())

    if trim_quantile is None:
        rho_trim = rho_full
    else:
        q = np.quantile(dists, trim_quantile)
        mask = dists <= q
        if not np.any(mask):
            rho_trim = rho_full
        else:
            rho_trim = float(dists[mask].mean())

    return rho_full, rho_trim

def estimate_L_gamma_feature_global() -> float:
    with torch.no_grad():
        W = model.classifier.weight.detach()
        diff = W.unsqueeze(0) - W.unsqueeze(1)
        norms = diff.norm(p=2, dim=2)
        Kc = norms.size(0)
        mask = ~torch.eye(Kc, dtype=torch.bool, device=norms.device)
        L_gamma = norms[mask].max().item()
    return float(L_gamma)

@torch.no_grad()
def hinge_loss_on_loader(loader: DataLoader, sigma: float) -> float:
    model.eval()
    total_hinge = 0.0
    total_n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        _, hinge = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        bs = x.size(0)
        total_hinge += hinge.sum().item()
        total_n += bs
    return total_hinge / max(total_n, 1)

@torch.no_grad()
def estimate_margin_drift(
    loader: DataLoader,
    sigma: float,
    max_batches: int = None,
    trim_quantile: float = 0.9,
):
    model.eval()
    all_diffs = []

    for b, (x, y) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits_P = model.logits_from_x_nograd(x)
        margin_P, _ = samplewise_margin_and_hinge(logits_P, y, margin=MARGIN)

        x_shift = apply_Q_transform(x.clone(), sigma)
        logits_Q = model.logits_from_x_nograd(x_shift)
        margin_Q, _ = samplewise_margin_and_hinge(logits_Q, y, margin=MARGIN)

        diff = (margin_Q - margin_P).abs().detach().cpu().numpy()
        all_diffs.append(diff)

    if not all_diffs:
        return float("nan"), float("nan")

    diffs = np.concatenate(all_diffs, axis=0)
    delta_full = float(diffs.mean())

    if trim_quantile is None:
        delta_trim = delta_full
    else:
        q = np.quantile(diffs, trim_quantile)
        mask = diffs <= q
        if not np.any(mask):
            delta_trim = delta_full
        else:
            delta_trim = float(diffs[mask].mean())

    return delta_full, delta_trim

def estimate_density_sup(scores: np.ndarray, nbins: int = 50) -> float:
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return float("nan")
    lo, hi = scores.min(), scores.max()
    if not np.isfinite(lo) or not np.isfinite(hi):
        return float("nan")
    if hi == lo:
        return 1.0
    pad = 1e-6 * (hi - lo)
    lo -= pad
    hi += pad
    hist, edges = np.histogram(scores, bins=nbins, range=(lo, hi))
    bin_width = edges[1] - edges[0]
    pdf_est = hist / (scores.size * bin_width)
    return float(pdf_est.max())

def empirical_W1_1d(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(a.size, b.size)
    if n == 0:
        return float("nan")
    a_sorted = np.sort(a)[:n]
    b_sorted = np.sort(b)[:n]
    return float(np.mean(np.abs(a_sorted - b_sorted)))

sigma_vals  = SIGMA_GRID.astype(float)
nominal_cov = 1.0 - ALPHA

scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
q_source = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

cov_source_Q = []
cov_hard_Q   = []
cov_alg_Q    = cov_Q_uP_epochs_sigma[-1, :].copy()

print("Computing source and hard pseudo calibration coverage vs σ ...")

for sigma in sigma_vals:
    sigma_f = float(sigma)

    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)

    cov_src = coverage_given_threshold(scores_Q_tst_true, q_source)
    cov_source_Q.append(cov_src)

    scores_Q_cal_hard = collect_pseudo_scores_hard(
        loader=Q_cal_loader,
        sigma=sigma_f,
        num_classes=NUM_CLASSES,
    )
    q_hard = conformal_quantile(scores_Q_cal_hard, ALPHA, finite_sample=True)
    cov_hard = coverage_given_threshold(scores_Q_tst_true, q_hard)
    cov_hard_Q.append(cov_hard)

cov_source_Q = np.array(cov_source_Q, dtype=np.float64)
cov_hard_Q   = np.array(cov_hard_Q, dtype=np.float64)

L_h_source = float(hinge_P_epochs[-1])
print(f"Final source hinge loss L_h(f,P) ≈ {L_h_source:.4f}")

print("Estimating margin drift δ_mix(σ) on P (full & trimmed) ...")
delta_margin_vals = []
delta_margin_trim_vals = []

for sigma in sigma_vals:
    delta_full, delta_trim = estimate_margin_drift(
        loader=P_cal_loader,
        sigma=float(sigma),
        max_batches=None,
        trim_quantile=0.9,
    )
    delta_margin_vals.append(delta_full)
    delta_margin_trim_vals.append(delta_trim)

delta_margin_vals = np.array(delta_margin_vals, dtype=np.float64)
delta_margin_trim_vals = np.array(delta_margin_trim_vals, dtype=np.float64)
print("δ_mix(σ) full:   ", delta_margin_vals)
print("δ_mix(σ) trimmed:", delta_margin_trim_vals)

print("Estimating latent ρ_mix(σ) (full & trimmed) ...")
rho_feat_vals = []
rho_feat_trim_vals = []

for sigma in sigma_vals:
    rho_full, rho_trim = estimate_feature_rho_mix(
        loader=P_cal_loader,
        sigma=float(sigma),
        max_batches=None,
        trim_quantile=0.9,
    )
    rho_feat_vals.append(rho_full)
    rho_feat_trim_vals.append(rho_trim)

rho_feat_vals = np.array(rho_feat_vals, dtype=np.float64)
rho_feat_trim_vals = np.array(rho_feat_trim_vals, dtype=np.float64)
print("ρ_mix(σ) full:   ", rho_feat_vals)
print("ρ_mix(σ) trimmed:", rho_feat_trim_vals)

L_gamma_global = estimate_L_gamma_feature_global()
print(f"Global feature-space margin Lipschitz L_γ ≈ {L_gamma_global:.4f}")

print("Estimating hinge loss L_h(f,Q_σ) for each σ ...")
L_h_Q_sigmas = []
for sigma in sigma_vals:
    L_h_Q = hinge_loss_on_loader(Q_tst_loader, sigma=float(sigma))
    L_h_Q_sigmas.append(L_h_Q)
L_h_Q_sigmas = np.array(L_h_Q_sigmas, dtype=np.float64)
print("L_h(f,Q_σ):", L_h_Q_sigmas)

cov_lb_W1 = []
M_vals    = []
W1_vals   = []

print("Estimating score-space W1 and density sup M(σ) on Q_cal (oracle) ...")
for sigma in sigma_vals:
    sigma_f = float(sigma)

    scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma=sigma_f)
    scores_Q_cal_pseudo = collect_pseudo_scores_hard(
        loader=Q_cal_loader,
        sigma=sigma_f,
        num_classes=NUM_CLASSES,
    )

    M_hat = estimate_density_sup(scores_Q_cal_pseudo, nbins=50)
    W1_emp = empirical_W1_1d(scores_Q_cal_pseudo, scores_Q_cal_true)

    M_vals.append(M_hat)
    W1_vals.append(W1_emp)

    gap = 2.0 * math.sqrt(max(0.0, M_hat * W1_emp))
    lb  = nominal_cov - gap
    lb  = max(0.0, min(1.0, lb))
    cov_lb_W1.append(lb)

cov_lb_W1 = np.array(cov_lb_W1, dtype=np.float64)
M_vals    = np.array(M_vals, dtype=np.float64)
W1_vals   = np.array(W1_vals, dtype=np.float64)

print("Estimated M(σ)  =", M_vals)
print("Estimated W1(σ) =", W1_vals)

tau_const = 0.0

cov_lb_oracle = nominal_cov - L_h_Q_sigmas
cov_lb_oracle = np.clip(cov_lb_oracle, 0.0, 1.0)

cov_lb_margin_tight = nominal_cov - (L_h_source + delta_margin_trim_vals)
cov_lb_margin_tight = np.clip(cov_lb_margin_tight, 0.0, 1.0)

cov_lb_rho_tight = nominal_cov - (L_h_source + L_gamma_global * rho_feat_trim_vals)
cov_lb_rho_tight = np.clip(cov_lb_rho_tight, 0.0, 1.0)

In [ ]:
import csv
import math
import random
from contextlib import contextmanager

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import scienceplots

    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

DATASET_TAG = "cifar10"
DATASET_TITLE = "CIFAR-10"
SIGMA_ALPHA_SWEEP = float(1.0)
ALPHA_GRID = np.round(np.linspace(0.05, 0.40, 8), 2)
N_U_POINTS = 30
N_REPEATS = 3
BASE_SEED = int(SEED) if "SEED" in globals() else 42

mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
plt.rcParams["lines.linewidth"] = 1.8

@contextmanager
def _temporary_seed(seed: int):
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)

def _sigma_tag(x: float) -> str:
    return f"{x:.2f}".replace(".", "p")

def _entropy_from_logits_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    probs /= probs.sum(axis=1, keepdims=True)
    return -(probs * np.log(probs + 1e-12)).sum(axis=1)

def _scores_for_labels_from_logits_np(logits: np.ndarray, labels: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    rows = np.arange(logits.shape[0])
    masked = logits.copy()
    masked[rows, labels] = -np.inf
    max_other = masked.max(axis=1)
    return max_other - logits[rows, labels]

def _conformal_quantile_np(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    n = scores.size
    if n == 0:
        return float("nan")
    qlevel = math.ceil((n + 1) * (1.0 - float(alpha))) / n
    if qlevel > 1.0:
        return float("inf")
    try:
        return float(np.quantile(scores, qlevel, method="higher"))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation="higher"))

def _coverage_np(scores_true: np.ndarray, q: float) -> float:
    return float(np.mean(np.asarray(scores_true) <= float(q)))

def _expected_set_size_from_logits_np(logits: np.ndarray, q: float) -> float:
    logits = np.asarray(logits, dtype=np.float64)
    top_sorted = np.sort(logits, axis=1)
    top1 = top_sorted[:, -1][:, None]
    top2 = top_sorted[:, -2][:, None]
    argmax = np.argmax(logits, axis=1)
    classes = np.arange(logits.shape[1])[None, :]
    max_other = np.where(classes == argmax[:, None], top2, top1)
    scores_all = max_other - logits
    return float(np.mean((scores_all <= float(q)).sum(axis=1)))

@torch.no_grad()
def _collect_logits_and_labels(loader, sigma: float, seed: int):
    model.eval()
    all_logits = []
    all_labels = []
    with _temporary_seed(seed):
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            if float(sigma) != 0.0:
                x = apply_Q_transform(x, float(sigma))
            logits = model.logits_from_x_nograd(x)
            all_logits.append(logits.detach().cpu().numpy().astype(np.float32))
            all_labels.append(y.detach().cpu().numpy().astype(np.int64))
    return np.concatenate(all_logits, axis=0), np.concatenate(all_labels, axis=0)

def _sample_random_labels(n: int, num_classes: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(low=0, high=num_classes, size=n, dtype=np.int64)

def _compute_u_star_and_q(
    p_cal_logits: np.ndarray,
    p_cal_entropy: np.ndarray,
    p_tst_true_scores: np.ndarray,
    rand_labels_p: np.ndarray,
    alpha: float,
    num_u_points: int,
):
    t_min = float(np.quantile(p_cal_entropy, 0.05))
    t_max = float(np.quantile(p_cal_entropy, 0.95))
    if np.isclose(t_min, t_max):
        u_grid = np.array([t_min], dtype=np.float64)
    else:
        u_grid = np.linspace(t_min, t_max, num_u_points)

    hard_labels = np.argmax(p_cal_logits, axis=1)
    target_cov = 1.0 - float(alpha)

    best_u = float(u_grid[0])
    best_q = None

    for u in u_grid:
        pseudo_labels = np.where(p_cal_entropy <= u, hard_labels, rand_labels_p)
        pseudo_scores = _scores_for_labels_from_logits_np(p_cal_logits, pseudo_labels)
        q_u = _conformal_quantile_np(pseudo_scores, alpha)
        cov_u = _coverage_np(p_tst_true_scores, q_u)
        if cov_u >= target_cov:
            best_u = float(u)
            best_q = float(q_u)

    if best_q is None:
        best_u = float(u_grid[0])
        pseudo_labels = np.where(p_cal_entropy <= best_u, hard_labels, rand_labels_p)
        pseudo_scores = _scores_for_labels_from_logits_np(p_cal_logits, pseudo_labels)
        best_q = _conformal_quantile_np(pseudo_scores, alpha)

    return best_u, float(best_q)

def _summarize(metric_matrix: np.ndarray):
    mean = metric_matrix.mean(axis=0)
    if metric_matrix.shape[0] > 1:
        std = metric_matrix.std(axis=0, ddof=1)
    else:
        std = np.zeros(metric_matrix.shape[1], dtype=np.float64)
    return mean, std

print(f"Running {DATASET_TITLE} alpha sweep at sigma={SIGMA_ALPHA_SWEEP:.2f}")
print(f"Alpha grid: {ALPHA_GRID.tolist()} | repeats: {N_REPEATS}")

P_cal_logits, P_cal_labels = _collect_logits_and_labels(P_cal_loader, sigma=0.0, seed=BASE_SEED + 11)
P_tst_logits, P_tst_labels = _collect_logits_and_labels(P_tst_loader, sigma=0.0, seed=BASE_SEED + 12)
P_cal_true_scores = _scores_for_labels_from_logits_np(P_cal_logits, P_cal_labels)
P_tst_true_scores = _scores_for_labels_from_logits_np(P_tst_logits, P_tst_labels)
P_cal_entropy = _entropy_from_logits_np(P_cal_logits)

method_labels = {
    "source": "Source Cal.",
    "hard": "Hard Pseudo-Cal.",
    "tuned": r"Source-tuned Pseudo-Cal. ($u_P^*$)",
    "oracle": "Target Cal. (Oracle)",
}
method_colors = {
    "source": "tab:red",
    "hard": "tab:blue",
    "tuned": "tab:green",
    "oracle": "tab:cyan",
}

coverage_store = {
    key: np.zeros((N_REPEATS, len(ALPHA_GRID)), dtype=np.float64)
    for key in method_labels
}
ess_store = {
    key: np.zeros((N_REPEATS, len(ALPHA_GRID)), dtype=np.float64)
    for key in method_labels
}
u_star_store = np.zeros((N_REPEATS, len(ALPHA_GRID)), dtype=np.float64)

for rep in range(N_REPEATS):
    rep_seed = BASE_SEED + 1000 * (rep + 1)

    Q_cal_logits, Q_cal_labels = _collect_logits_and_labels(
        Q_cal_loader,
        sigma=SIGMA_ALPHA_SWEEP,
        seed=rep_seed + 1,
    )
    Q_tst_logits, Q_tst_labels = _collect_logits_and_labels(
        Q_tst_loader,
        sigma=SIGMA_ALPHA_SWEEP,
        seed=rep_seed + 2,
    )

    Q_cal_true_scores = _scores_for_labels_from_logits_np(Q_cal_logits, Q_cal_labels)
    Q_tst_true_scores = _scores_for_labels_from_logits_np(Q_tst_logits, Q_tst_labels)
    Q_cal_entropy = _entropy_from_logits_np(Q_cal_logits)
    Q_cal_hard_labels = np.argmax(Q_cal_logits, axis=1)
    Q_cal_hard_scores = _scores_for_labels_from_logits_np(Q_cal_logits, Q_cal_hard_labels)

    rand_labels_p = _sample_random_labels(len(P_cal_logits), NUM_CLASSES, seed=rep_seed + 3)
    rand_labels_q = _sample_random_labels(len(Q_cal_logits), NUM_CLASSES, seed=rep_seed + 4)

    for a_idx, alpha in enumerate(ALPHA_GRID):
        q_source = _conformal_quantile_np(P_cal_true_scores, alpha)
        q_hard = _conformal_quantile_np(Q_cal_hard_scores, alpha)
        q_oracle = _conformal_quantile_np(Q_cal_true_scores, alpha)

        u_star, _ = _compute_u_star_and_q(
            p_cal_logits=P_cal_logits,
            p_cal_entropy=P_cal_entropy,
            p_tst_true_scores=P_tst_true_scores,
            rand_labels_p=rand_labels_p,
            alpha=float(alpha),
            num_u_points=N_U_POINTS,
        )
        u_star_store[rep, a_idx] = u_star

        q_cal_pseudo_labels = np.where(Q_cal_entropy <= u_star, Q_cal_hard_labels, rand_labels_q)
        q_cal_pseudo_scores = _scores_for_labels_from_logits_np(Q_cal_logits, q_cal_pseudo_labels)
        q_tuned = _conformal_quantile_np(q_cal_pseudo_scores, alpha)

        for method_key, q_value in {
            "source": q_source,
            "hard": q_hard,
            "tuned": q_tuned,
            "oracle": q_oracle,
        }.items():
            coverage_store[method_key][rep, a_idx] = _coverage_np(Q_tst_true_scores, q_value)
            ess_store[method_key][rep, a_idx] = _expected_set_size_from_logits_np(Q_tst_logits, q_value)

    print(f"Completed repeat {rep + 1}/{N_REPEATS}")

summary_rows = []
for a_idx, alpha in enumerate(ALPHA_GRID):
    row = {
        "alpha": float(alpha),
        "nominal_coverage": float(1.0 - alpha),
    }
    u_mean, u_std = _summarize(u_star_store[:, [a_idx]])
    row["u_P_star_mean"] = float(u_mean[0])
    row["u_P_star_std"] = float(u_std[0])

    for method_key in method_labels:
        cov_mean, cov_std = _summarize(coverage_store[method_key][:, [a_idx]])
        ess_mean, ess_std = _summarize(ess_store[method_key][:, [a_idx]])
        row[f"{method_key}_coverage_mean"] = float(cov_mean[0])
        row[f"{method_key}_coverage_std"] = float(cov_std[0])
        row[f"{method_key}_ess_mean"] = float(ess_mean[0])
        row[f"{method_key}_ess_std"] = float(ess_std[0])
    summary_rows.append(row)

sigma_str = _sigma_tag(SIGMA_ALPHA_SWEEP)
csv_path = f"{DATASET_TAG}_alpha_sweep_sigma_{sigma_str}.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

fig_cov, ax_cov = plt.subplots(figsize=(3.5, 3.5))
ax_cov.plot(
    ALPHA_GRID,
    1.0 - ALPHA_GRID,
    linestyle="--",
    color="0.45",
    linewidth=1.6,
    label=r"Nominal $1-\alpha$",
)

for method_key, method_label in method_labels.items():
    mean_vals, std_vals = _summarize(coverage_store[method_key])
    ax_cov.plot(
        ALPHA_GRID,
        mean_vals,
        marker="o",
        markersize=4.5,
        color=method_colors[method_key],
        label=method_label,
    )
    if N_REPEATS > 1:
        ax_cov.fill_between(
            ALPHA_GRID,
            np.clip(mean_vals - std_vals, 0.0, 1.0),
            np.clip(mean_vals + std_vals, 0.0, 1.0),
            color=method_colors[method_key],
            alpha=0.12,
        )

ax_cov.set_xlabel(r"Miscoverage level $\alpha$")
ax_cov.set_ylabel(r"Empirical coverage on $Q_\sigma$")
ax_cov.set_title(fr"{DATASET_TITLE}: coverage vs. $\alpha$ at $\sigma={SIGMA_ALPHA_SWEEP:.2f}$")
ax_cov.set_xlim(ALPHA_GRID.min() - 0.01, ALPHA_GRID.max() + 0.01)
ax_cov.set_ylim(0.4, 1.02)
ax_cov.grid(True, linewidth=0.5, alpha=0.5)
ax_cov.legend(loc="best", frameon=False)
fig_cov.tight_layout()

cov_pdf = f"{DATASET_TAG}_coverage_vs_alpha_sigma_{sigma_str}.pdf"
cov_png = f"{DATASET_TAG}_coverage_vs_alpha_sigma_{sigma_str}.png"
fig_cov.savefig(cov_pdf, bbox_inches="tight")
fig_cov.savefig(cov_png, dpi=300, bbox_inches="tight")
plt.show()

fig_ess, ax_ess = plt.subplots(figsize=(3.5, 3.5))
for method_key, method_label in method_labels.items():
    mean_vals, std_vals = _summarize(ess_store[method_key])
    ax_ess.plot(
        ALPHA_GRID,
        mean_vals,
        marker="o",
        markersize=4.5,
        color=method_colors[method_key],
        label=method_label,
    )
    if N_REPEATS > 1:
        ax_ess.fill_between(
            ALPHA_GRID,
            np.maximum(mean_vals - std_vals, 0.0),
            mean_vals + std_vals,
            color=method_colors[method_key],
            alpha=0.12,
        )

ax_ess.set_xlabel(r"Miscoverage level $\alpha$")
ax_ess.set_ylabel(r"Expected set size on $Q_\sigma$")
ax_ess.set_title(fr"{DATASET_TITLE}: ESS vs. $\alpha$ at $\sigma={SIGMA_ALPHA_SWEEP:.2f}$")
ax_ess.set_xlim(ALPHA_GRID.min() - 0.01, ALPHA_GRID.max() + 0.01)
ax_ess.set_ylim(bottom=0.3)
ax_ess.grid(True, linewidth=0.5, alpha=0.5)
ax_ess.legend(loc="best", frameon=False)
fig_ess.tight_layout()

ess_pdf = f"{DATASET_TAG}_ess_vs_alpha_sigma_{sigma_str}.pdf"
ess_png = f"{DATASET_TAG}_ess_vs_alpha_sigma_{sigma_str}.png"
fig_ess.savefig(ess_pdf, bbox_inches="tight")
fig_ess.savefig(ess_png, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved summary CSV to {csv_path}")
print(f"Saved coverage plot to {cov_pdf} and {cov_png}")
print(f"Saved ESS plot to {ess_pdf} and {ess_png}")

In [ ]:
import csv
import math
import os
import random
from contextlib import contextmanager

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.datasets.utils import download_and_extract_archive

try:
    import scienceplots

    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

CIFAR10C_CORRUPTION = "shot_noise"
SEVERITY_LEVELS = np.array([0, 1, 2, 3, 4, 5], dtype=int)
N_REPEATS = 3
N_U_POINTS = 30
BASE_SEED = int(SEED) if "SEED" in globals() else 42
DATASET_TAG = f"cifar10_cifar10c_{CIFAR10C_CORRUPTION}"
DATASET_TITLE = f"CIFAR-10-C"
INFER_BATCH_SIZE = int(BATCH_SIZE) if "BATCH_SIZE" in globals() else 128

CIFAR10C_ROOT = os.path.join(".", "data")
CIFAR10C_URL = "https://zenodo.org/record/2535967/files/CIFAR-10-C.tar"
CIFAR10C_MD5 = "56bf5dcef84df0e2308c6dcbcbbd8499"
CIFAR10C_BASE_FOLDER = os.path.join(CIFAR10C_ROOT, "CIFAR-10-C")

CIFAR10_MEAN_ARRAY = np.asarray(CIFAR10_MEAN, dtype=np.float32).reshape(1, 1, 1, 3)
CIFAR10_STD_ARRAY = np.asarray(CIFAR10_STD, dtype=np.float32).reshape(1, 1, 1, 3)

mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
plt.rcParams["lines.linewidth"] = 1.8

COLOR_SOURCE = "tab:red"
COLOR_HARD = "tab:blue"
COLOR_TAU = "tab:orange"
COLOR_TUNED = "tab:green"
COLOR_TARGET = "black"
COLOR_BOUND = "0.25"
COLOR_NOMINAL = "0.45"

@contextmanager
def _temporary_seed(seed: int):
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)

def _entropy_from_logits_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    probs /= probs.sum(axis=1, keepdims=True)
    return -(probs * np.log(probs + 1e-12)).sum(axis=1)

def _scores_for_labels_from_logits_np(logits: np.ndarray, labels: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    rows = np.arange(logits.shape[0])
    masked = logits.copy()
    masked[rows, labels] = -np.inf
    max_other = masked.max(axis=1)
    return max_other - logits[rows, labels]

def _conformal_quantile_np(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    n = scores.size
    if n == 0:
        return float("nan")
    qlevel = math.ceil((n + 1) * (1.0 - float(alpha))) / n
    if qlevel > 1.0:
        return float("inf")
    try:
        return float(np.quantile(scores, qlevel, method="higher"))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation="higher"))

def _coverage_np(scores_true: np.ndarray, q: float) -> float:
    return float(np.mean(np.asarray(scores_true) <= float(q)))

def _expected_set_size_from_logits_np(logits: np.ndarray, q: float) -> float:
    logits = np.asarray(logits, dtype=np.float64)
    top_sorted = np.sort(logits, axis=1)
    top1 = top_sorted[:, -1][:, None]
    top2 = top_sorted[:, -2][:, None]
    argmax = np.argmax(logits, axis=1)
    classes = np.arange(logits.shape[1])[None, :]
    max_other = np.where(classes == argmax[:, None], top2, top1)
    scores_all = max_other - logits
    return float(np.mean((scores_all <= float(q)).sum(axis=1)))

def _samplewise_margins_np(logits: np.ndarray, labels: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    rows = np.arange(logits.shape[0])
    masked = logits.copy()
    masked[rows, labels] = -np.inf
    max_other = masked.max(axis=1)
    return logits[rows, labels] - max_other

def _hinge_loss_from_margins_np(margins: np.ndarray) -> float:
    if margins.size == 0:
        return 0.0
    return float(np.mean(np.maximum(1.0 - margins, 0.0)))

def _ramp_loss_from_margins_np(margins: np.ndarray) -> float:
    if margins.size == 0:
        return 0.0
    hinge = np.maximum(1.0 - margins, 0.0)
    ramp = np.minimum(hinge, 1.0)
    return float(np.mean(ramp))

def _ensure_cifar10c_downloaded():
    corruption_path = os.path.join(CIFAR10C_BASE_FOLDER, f"{CIFAR10C_CORRUPTION}.npy")
    labels_path = os.path.join(CIFAR10C_BASE_FOLDER, "labels.npy")

    if os.path.exists(corruption_path) and os.path.exists(labels_path):
        return corruption_path, labels_path

    print("Downloading/extracting official CIFAR-10-C dataset ...")
    download_and_extract_archive(
        url=CIFAR10C_URL,
        download_root=CIFAR10C_ROOT,
        md5=CIFAR10C_MD5,
        remove_finished=True,
    )

    if not (os.path.exists(corruption_path) and os.path.exists(labels_path)):
        raise RuntimeError("CIFAR-10-C download/extraction failed or files are missing.")

    return corruption_path, labels_path

def _load_cifar10c_severity_arrays(severity: int):
    if severity == 0:
        raise ValueError("Severity 0 should use the clean CIFAR-10 source/test loaders, not CIFAR-10-C arrays.")

    corruption_path, labels_path = _ensure_cifar10c_downloaded()
    all_images = np.load(corruption_path, mmap_mode="r")
    labels = np.load(labels_path)

    start = (int(severity) - 1) * 10000
    end = int(severity) * 10000
    images = np.asarray(all_images[start:end])
    if labels.shape[0] == 10000:
        severity_labels = labels
    elif labels.shape[0] == 50000:
        severity_labels = np.asarray(labels[start:end])
    else:
        raise RuntimeError(
            "Unexpected CIFAR-10-C label shape. "
            f"Expected 10000 or 50000 labels, got {labels.shape}."
        )

    if images.shape[0] != 10000 or severity_labels.shape[0] != 10000:
        raise RuntimeError(
            "Unexpected CIFAR-10-C severity slice shapes: "
            f"images={images.shape}, labels={severity_labels.shape}, severity={severity}"
        )

    idx_cal = np.arange(0, 5000)
    idx_tst = np.arange(5000, 10000)

    return (
        images[idx_cal],
        severity_labels[idx_cal].astype(np.int64),
        images[idx_tst],
        severity_labels[idx_tst].astype(np.int64),
    )

def _cifar10c_uint8_to_normalized_nchw(x_uint8: np.ndarray) -> np.ndarray:
    x = x_uint8.astype(np.float32) / 255.0
    x = (x - CIFAR10_MEAN_ARRAY) / CIFAR10_STD_ARRAY
    x = np.transpose(x, (0, 3, 1, 2))
    return x.astype(np.float32)

@torch.no_grad()
def _collect_logits_from_numpy_images(images_uint8: np.ndarray, labels: np.ndarray):
    model.eval()
    x_nchw = _cifar10c_uint8_to_normalized_nchw(images_uint8)
    labels = np.asarray(labels, dtype=np.int64)

    all_logits = []
    for start in range(0, x_nchw.shape[0], INFER_BATCH_SIZE):
        end = min(start + INFER_BATCH_SIZE, x_nchw.shape[0])
        xb = torch.from_numpy(x_nchw[start:end]).to(DEVICE)
        logits = model.logits_from_x_nograd(xb)
        all_logits.append(logits.detach().cpu().numpy().astype(np.float32))

    return np.concatenate(all_logits, axis=0), labels

@torch.no_grad()
def _collect_clean_logits_and_labels(loader):
    model.eval()
    all_logits = []
    all_labels = []

    for x, y in loader:
        x = x.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        all_logits.append(logits.detach().cpu().numpy().astype(np.float32))
        all_labels.append(y.detach().cpu().numpy().astype(np.int64))

    return np.concatenate(all_logits, axis=0), np.concatenate(all_labels, axis=0)

def _sample_random_labels(n: int, num_classes: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(low=0, high=num_classes, size=n, dtype=np.int64)

def _compute_u_star_and_q(
    p_cal_logits: np.ndarray,
    p_cal_entropy: np.ndarray,
    p_tst_true_scores: np.ndarray,
    rand_labels_p: np.ndarray,
    alpha: float,
    num_u_points: int,
):
    t_min = float(np.quantile(p_cal_entropy, 0.05))
    t_max = float(np.quantile(p_cal_entropy, 0.95))
    if np.isclose(t_min, t_max):
        u_grid = np.array([t_min], dtype=np.float64)
    else:
        u_grid = np.linspace(t_min, t_max, num_u_points)

    hard_labels = np.argmax(p_cal_logits, axis=1)
    target_cov = 1.0 - float(alpha)

    best_u = float(u_grid[0])
    best_q = None

    for u in u_grid:
        pseudo_labels = np.where(p_cal_entropy <= u, hard_labels, rand_labels_p)
        pseudo_scores = _scores_for_labels_from_logits_np(p_cal_logits, pseudo_labels)
        q_u = _conformal_quantile_np(pseudo_scores, alpha)
        cov_u = _coverage_np(p_tst_true_scores, q_u)
        if cov_u >= target_cov:
            best_u = float(u)
            best_q = float(q_u)

    if best_q is None:
        best_u = float(u_grid[0])
        pseudo_labels = np.where(p_cal_entropy <= best_u, hard_labels, rand_labels_p)
        pseudo_scores = _scores_for_labels_from_logits_np(p_cal_logits, pseudo_labels)
        best_q = _conformal_quantile_np(pseudo_scores, alpha)

    return best_u, float(best_q)

def _summarize(metric_matrix: np.ndarray):
    mean = metric_matrix.mean(axis=0)
    if metric_matrix.shape[0] > 1:
        std = metric_matrix.std(axis=0, ddof=1)
    else:
        std = np.zeros(metric_matrix.shape[1], dtype=np.float64)
    return mean, std

print("=" * 80)
print(f"CIFAR-10-C severity sweep using official corruption: {CIFAR10C_CORRUPTION}")
print("Severity 0 = clean CIFAR-10; severities 1..5 are official CIFAR-10-C slices.")
print("=" * 80)

P_cal_logits, P_cal_labels = _collect_clean_logits_and_labels(P_cal_loader)
P_tst_logits, P_tst_labels = _collect_clean_logits_and_labels(P_tst_loader)
P_cal_true_scores = _scores_for_labels_from_logits_np(P_cal_logits, P_cal_labels)
P_tst_true_scores = _scores_for_labels_from_logits_np(P_tst_logits, P_tst_labels)
P_cal_entropy = _entropy_from_logits_np(P_cal_logits)
nominal_cov = 1.0 - float(ALPHA)
EPS = 1e-12

coverage_plot_labels = {
    "source": "Source Cal.",
    "oracle": "Target Cal.",
    "hard": "Hard Pseudo-Cal.",
    "tuned": "Source-Tuned Pseudo-Cal.",
}
coverage_plot_colors = {
    "source": COLOR_SOURCE,
    "oracle": COLOR_TARGET,
    "hard": COLOR_HARD,
    "tuned": COLOR_TUNED,
}
coverage_plot_order = ["source", "oracle", "hard", "tuned"]

ess_plot_labels = {
    "source": r"Source Cal. ($q_{P,\alpha}$)",
    "hard": r"Hard Pseudo-Cal. ($q_{\tilde{Q}_{\mathrm{severity}},\alpha}$)",
    "tuned": r"Source-Tuned Pseudo-Cal. ($q_{\tilde{Q}_{\mathrm{severity}},\alpha}^{u_P^\star}$)",
    "tau": r"Hard Pseudo-Cal. + $\tau$ ($q_{\tilde{Q}_{\mathrm{severity}},\alpha}+\tau$)",
    "oracle": r"Target Cal. ($q_{Q_{\mathrm{severity}},\alpha}$)",
}
ess_plot_colors = {
    "source": COLOR_SOURCE,
    "hard": COLOR_HARD,
    "tuned": COLOR_TUNED,
    "tau": COLOR_TAU,
    "oracle": COLOR_TARGET,
}
ess_plot_order = ["source", "hard", "tuned", "tau", "oracle"]
stored_method_keys = ["source", "hard", "tuned", "tau", "oracle"]

P_cal_hard_labels = np.argmax(P_cal_logits, axis=1)
P_cal_hard_scores = _scores_for_labels_from_logits_np(P_cal_logits, P_cal_hard_labels)
q_pseudo_P = _conformal_quantile_np(P_cal_hard_scores, ALPHA)
coverage_P_pseudo = _coverage_np(P_tst_true_scores, q_pseudo_P)
Delta_P = float(nominal_cov - coverage_P_pseudo)
P_tst_margins = _samplewise_margins_np(P_tst_logits, P_tst_labels)
L_h_P = _hinge_loss_from_margins_np(P_tst_margins)
denom_base = float(L_h_P - Delta_P)

print(
    f"Source-side tau ingredients: q_pseudo_P={q_pseudo_P:.6f}, "
    f"coverage_P_pseudo={coverage_P_pseudo:.6f}, Delta_P={Delta_P:.6f}, "
    f"L_h(P)={L_h_P:.6f}, denom={denom_base:.6f}"
)

coverage_store = {
    key: np.zeros((N_REPEATS, len(SEVERITY_LEVELS)), dtype=np.float64)
    for key in stored_method_keys
}
ess_store = {
    key: np.zeros((N_REPEATS, len(SEVERITY_LEVELS)), dtype=np.float64)
    for key in stored_method_keys
}
u_star_store = np.zeros((N_REPEATS, len(SEVERITY_LEVELS)), dtype=np.float64)
tau_store = np.zeros((N_REPEATS, len(SEVERITY_LEVELS)), dtype=np.float64)
bound_store = {
    "lb_ramp_q": np.zeros((N_REPEATS, len(SEVERITY_LEVELS)), dtype=np.float64),
}

target_cache = {}

for rep in range(N_REPEATS):
    rep_seed = BASE_SEED + 1000 * (rep + 1)
    rand_labels_p = _sample_random_labels(len(P_cal_logits), NUM_CLASSES, seed=rep_seed + 3)

    u_star, _ = _compute_u_star_and_q(
        p_cal_logits=P_cal_logits,
        p_cal_entropy=P_cal_entropy,
        p_tst_true_scores=P_tst_true_scores,
        rand_labels_p=rand_labels_p,
        alpha=float(ALPHA),
        num_u_points=N_U_POINTS,
    )

    q_source = _conformal_quantile_np(P_cal_true_scores, ALPHA)

    for s_idx, severity in enumerate(SEVERITY_LEVELS):
        severity_int = int(severity)

        if severity_int == 0:
            Q_cal_logits, Q_cal_labels = P_cal_logits, P_cal_labels
            Q_tst_logits, Q_tst_labels = P_tst_logits, P_tst_labels
        else:
            if severity_int not in target_cache:
                cal_images, cal_labels, tst_images, tst_labels = _load_cifar10c_severity_arrays(severity_int)
                target_cache[severity_int] = (
                    _collect_logits_from_numpy_images(cal_images, cal_labels),
                    _collect_logits_from_numpy_images(tst_images, tst_labels),
                )

            (Q_cal_logits, Q_cal_labels), (Q_tst_logits, Q_tst_labels) = target_cache[severity_int]

        Q_cal_true_scores = _scores_for_labels_from_logits_np(Q_cal_logits, Q_cal_labels)
        Q_tst_true_scores = _scores_for_labels_from_logits_np(Q_tst_logits, Q_tst_labels)
        Q_cal_entropy = _entropy_from_logits_np(Q_cal_logits)
        Q_cal_hard_labels = np.argmax(Q_cal_logits, axis=1)
        Q_cal_hard_scores = _scores_for_labels_from_logits_np(Q_cal_logits, Q_cal_hard_labels)
        rand_labels_q = _sample_random_labels(len(Q_cal_logits), NUM_CLASSES, seed=rep_seed + 500 + 10 * severity_int)

        q_hard = _conformal_quantile_np(Q_cal_hard_scores, ALPHA)
        q_oracle = _conformal_quantile_np(Q_cal_true_scores, ALPHA)
        q_tuned_labels = np.where(Q_cal_entropy <= u_star, Q_cal_hard_labels, rand_labels_q)
        q_tuned_scores = _scores_for_labels_from_logits_np(Q_cal_logits, q_tuned_labels)
        q_tuned = _conformal_quantile_np(q_tuned_scores, ALPHA)
        Q_tst_margins = _samplewise_margins_np(Q_tst_logits, Q_tst_labels)
        L_r_Q = _ramp_loss_from_margins_np(Q_tst_margins)
        bound_store["lb_ramp_q"][rep, s_idx] = np.clip(nominal_cov - L_r_Q, 0.0, 1.0)

        L_h_Q = _hinge_loss_from_margins_np(Q_tst_margins)
        if abs(denom_base) <= EPS:
            tau_raw = 0.0
        else:
            tau_raw = 2.0 * (L_h_Q / denom_base - 1.0)
        tau = float(max(0.0, tau_raw))
        q_tau = q_hard + tau

        for method_key, q_value in {
            "source": q_source,
            "hard": q_hard,
            "tuned": q_tuned,
            "tau": q_tau,
            "oracle": q_oracle,
        }.items():
            coverage_store[method_key][rep, s_idx] = _coverage_np(Q_tst_true_scores, q_value)
            ess_store[method_key][rep, s_idx] = _expected_set_size_from_logits_np(Q_tst_logits, q_value)

        u_star_store[rep, s_idx] = u_star
        tau_store[rep, s_idx] = tau

        print(
            f"repeat={rep + 1}/{N_REPEATS} | severity={severity_int} | "
            f"source={coverage_store['source'][rep, s_idx]:.4f} | "
            f"hard={coverage_store['hard'][rep, s_idx]:.4f} | "
            f"tau={coverage_store['tau'][rep, s_idx]:.4f} | "
            f"tuned={coverage_store['tuned'][rep, s_idx]:.4f} | "
            f"oracle={coverage_store['oracle'][rep, s_idx]:.4f} | "
            f"LB_ramp={bound_store['lb_ramp_q'][rep, s_idx]:.4f}"
        )

summary_rows = []
for s_idx, severity in enumerate(SEVERITY_LEVELS):
    row = {
        "severity": int(severity),
        "nominal_coverage": float(1.0 - ALPHA),
    }

    u_mean, u_std = _summarize(u_star_store[:, [s_idx]])
    row["u_P_star_mean"] = float(u_mean[0])
    row["u_P_star_std"] = float(u_std[0])

    tau_mean, tau_std = _summarize(tau_store[:, [s_idx]])
    row["tau_mean"] = float(tau_mean[0])
    row["tau_std"] = float(tau_std[0])

    lb_mean, lb_std = _summarize(bound_store["lb_ramp_q"][:, [s_idx]])
    row["lb_ramp_q_mean"] = float(lb_mean[0])
    row["lb_ramp_q_std"] = float(lb_std[0])

    for method_key in stored_method_keys:
        cov_mean, cov_std = _summarize(coverage_store[method_key][:, [s_idx]])
        ess_mean, ess_std = _summarize(ess_store[method_key][:, [s_idx]])
        row[f"{method_key}_coverage_mean"] = float(cov_mean[0])
        row[f"{method_key}_coverage_std"] = float(cov_std[0])
        row[f"{method_key}_ess_mean"] = float(ess_mean[0])
        row[f"{method_key}_ess_std"] = float(ess_std[0])
    summary_rows.append(row)

csv_path = f"{DATASET_TAG}_coverage_ess_vs_severity.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

fig_cov, ax_cov = plt.subplots(figsize=(3.5, 3.5))
legend_handles_cov = {}
legend_handles_cov["nominal"] = ax_cov.axhline(
    nominal_cov,
    linestyle="--",
    color=COLOR_NOMINAL,
    linewidth=1.6,
    label=r"$1-\alpha$",
)

for method_key in coverage_plot_order:
    method_label = coverage_plot_labels[method_key]
    mean_vals, std_vals = _summarize(coverage_store[method_key])
    (line_handle,) = ax_cov.plot(
        SEVERITY_LEVELS,
        mean_vals,
        marker="o",
        markersize=4.5,
        color=coverage_plot_colors[method_key],
        label=method_label,
    )
    legend_handles_cov[method_key] = line_handle
    if N_REPEATS > 1:
        ax_cov.fill_between(
            SEVERITY_LEVELS,
            np.clip(mean_vals - std_vals, 0.0, 1.0),
            np.clip(mean_vals + std_vals, 0.0, 1.0),
            color=coverage_plot_colors[method_key],
            alpha=0.12,
        )

lb_ramp_q_mean, _ = _summarize(bound_store["lb_ramp_q"])
legend_handles_cov["lb_ramp_q"] = ax_cov.plot(
    SEVERITY_LEVELS,
    lb_ramp_q_mean,
    color=COLOR_BOUND,
    linestyle=":",
    label=r"$1-\alpha - L_r(f,Q_{\mathrm{severity}})$",
)[0]

ax_cov.set_xlabel(f"Severity (0 = clean)")
ax_cov.set_ylabel("Empirical coverage on target")
ax_cov.set_title(f"{DATASET_TITLE}: Coverage vs. Severity (alpha={ALPHA:.2f})")
ax_cov.set_xticks(SEVERITY_LEVELS)
ax_cov.set_ylim(0.0, 1.02)
ax_cov.grid(True, linewidth=0.5, alpha=0.5)
ax_cov.legend(
    [
        legend_handles_cov["source"],
        legend_handles_cov["oracle"],
        legend_handles_cov["hard"],
        legend_handles_cov["tuned"],
        legend_handles_cov["nominal"],
        legend_handles_cov["lb_ramp_q"],
    ],
    [
        coverage_plot_labels["source"],
        coverage_plot_labels["oracle"],
        coverage_plot_labels["hard"],
        coverage_plot_labels["tuned"],
        r"$1-\alpha$",
        r"$1-\alpha - L_r(f,Q_{\mathrm{severity}})$",
    ],
    loc="best",
    frameon=False,
)
fig_cov.tight_layout()

cov_pdf = f"{DATASET_TAG}_coverage_vs_severity.pdf"
cov_png = f"{DATASET_TAG}_coverage_vs_severity.png"
fig_cov.savefig(cov_pdf, bbox_inches="tight")
fig_cov.savefig(cov_png, dpi=300, bbox_inches="tight")
plt.show()

fig_ess, ax_ess = plt.subplots(figsize=(3.5, 3.5))
for method_key in ess_plot_order:
    method_label = ess_plot_labels[method_key]
    mean_vals, std_vals = _summarize(ess_store[method_key])
    ax_ess.plot(
        SEVERITY_LEVELS,
        mean_vals,
        marker="o",
        markersize=4.5,
        color=ess_plot_colors[method_key],
        label=method_label,
    )
    if N_REPEATS > 1:
        ax_ess.fill_between(
            SEVERITY_LEVELS,
            np.maximum(mean_vals - std_vals, 0.0),
            mean_vals + std_vals,
            color=ess_plot_colors[method_key],
            alpha=0.12,
        )

ax_ess.set_xlabel(f"Severity (0 = clean)")
ax_ess.set_ylabel("Expected set size on target")
ax_ess.set_title(f"{DATASET_TITLE}: ESS vs. Severity (alpha={ALPHA:.2f})")
ax_ess.set_xticks(SEVERITY_LEVELS)
ax_ess.set_ylim(bottom=0.0)
ax_ess.grid(True, linewidth=0.5, alpha=0.5)
ax_ess.legend(loc="best", frameon=False)
fig_ess.tight_layout()

ess_pdf = f"{DATASET_TAG}_ess_vs_severity.pdf"
ess_png = f"{DATASET_TAG}_ess_vs_severity.png"
fig_ess.savefig(ess_pdf, bbox_inches="tight")
fig_ess.savefig(ess_png, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved summary CSV to {csv_path}")
print(f"Saved coverage plot to {cov_pdf} and {cov_png}")
print(f"Saved ESS plot to {ess_pdf} and {ess_png}")